In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import pydicom
from PIL import Image

# Converting to PNG

In [ ]:
CSV_PATH = " # patient-information"
INPUT_ROOT_NAME = " # expert-annotated-slices-path" 
OUTPUT_ROOT_NAME = " # converted-png-path" 

WINDOW_LEVEL = -700
WINDOW_WIDTH  = 1700

print("WL/WW:", WINDOW_LEVEL, WINDOW_WIDTH)

In [ ]:
df = pd.read_csv(CSV_PATH)
assert "slice_path" in df.columns, "CSV must contain a 'slice_path' column."

df["slice_path"] = df["slice_path"].astype(str).str.strip()
df = df[df["slice_path"].map(os.path.exists)].copy()
print(f"Converting {len(df)} slices...")

In [ ]:
# Map ".../selected_slice/…/file.dcm" -> ".../converted_png/…/file.png"
in_parts = INPUT_ROOT_NAME
out_parts = OUTPUT_ROOT_NAME

png_paths = []
for dcm_path in df["slice_path"]:
    p = Path(dcm_path)
    parts = list(p.parts)
    if in_parts in parts:
        parts[parts.index(in_parts)] = out_parts
        out_p = Path(*parts).with_suffix(".png")
    else:

        out_p = Path(out_parts) / p.with_suffix(".png").name
    png_paths.append(str(out_p))

df["png_path"] = png_paths

for out_p in df["png_path"]:
    Path(out_p).parent.mkdir(parents=True, exist_ok=True)

df.head()

In [ ]:
import pydicom
import numpy as np
from PIL import Image

n_ok = 0
n_fail = 0

low  = WINDOW_LEVEL - (WINDOW_WIDTH / 2.0)
high = WINDOW_LEVEL + (WINDOW_WIDTH / 2.0)

for dcm_path, png_path in zip(df["slice_path"], df["png_path"]):
    try:
        ds = pydicom.dcmread(dcm_path)
        arr = ds.pixel_array.astype(np.float32)

        # HU conversion
        slope = float(getattr(ds, "RescaleSlope", 1.0))
        intercept = float(getattr(ds, "RescaleIntercept", 0.0))
        hu = arr * slope + intercept

        # Windowing
        windowed = np.clip(hu, low, high)

        # Scale to 0–255
        img8 = ((windowed - low) / (high - low) * 255.0).astype(np.uint8)

        # Convert grayscale → RGB
        img_rgb = Image.fromarray(img8).convert("RGB")

        # Save RGB PNG
        img_rgb.save(png_path)

        n_ok += 1

    except Exception as e:
        print(f"[ERR] {dcm_path} -> {e}")
        n_fail += 1

print(f"Converted to RGB PNG: {n_ok} | Failed: {n_fail}")

In [ ]:
df["converted"] = df["png_path"].map(lambda p: os.path.exists(p))
manifest_path = "converted_slice.csv"
df.to_csv(manifest_path, index=False)
print(f"Saved manifest: {manifest_path}")
print(df["converted"].value_counts(dropna=False))

# Train Test Validation

In [ ]:
import pandas as pd

# Path to your CSV file
csv_path = " # png-slice-information"

# Read the CSV file
df = pd.read_csv(csv_path)

# Get unique pid values
unique_pids = df['pid'].unique()

# If you want to count how many unique ones
print("\nTotal unique PIDs:", len(unique_pids))

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Paths
src_path = Path(" # png-slice-information")          # change if needed
train_path = Path(" # train-png-slice-information")          # change if needed
test_path = Path(" # test-png-slice-information")          # change if needed

# Load
df = pd.read_csv(src_path)

pid_col = "pid"

# Unique PIDs with conversion check
pid_dtype = df[pid_col].dtype
print(f"Detected PID column: {pid_col} (dtype: {pid_dtype})")

# Only convert if dtype is numeric or if it looks like float-encoded IDs (e.g., 1234.0)
if np.issubdtype(pid_dtype, np.number):
    print("→ Numeric PID detected. Converting to string and cleaning possible '.0' suffixes.")
    pids_series = df[pid_col].dropna().astype(str).str.replace(r"\.0$", "", regex=True).str.strip()
else:
    # If already string-like, just clean whitespace and ensure no '.0' artifacts
    pids_series = df[pid_col].dropna().astype(str).str.strip().str.replace(r"\.0$", "", regex=True)

unique_pids = pids_series.unique()
n_unique = len(unique_pids)
print(f"Unique PIDs detected: {n_unique}")

# Desired counts
n_train, n_test = 670, 75
if n_unique < (n_train + n_test):
    raise ValueError(f"Not enough unique PIDs ({n_unique}) to split into {n_train} train and {n_test} test.")

# split
block_size = 10
train_pids = set()
test_pids = set()

for i in range(0, len(unique_pids), block_size):
    block = unique_pids[i:i + block_size]

    # First 9 → train
    train_pids.update(block[:9])

    # 10th → test (only if it exists)
    if len(block) == block_size:
        test_pids.add(block[9])

pid_as_str = pids_series
train_df = df[pid_as_str.isin(train_pids)].copy()
test_df  = df[pid_as_str.isin(test_pids)].copy()

train_df.to_csv(train_path, index=False)
test_df.to_csv(test_path, index=False)

# Sanity checks
print("First 20 unique PIDs:", unique_pids[:20])
print("Train PIDs from first 20:", [pid for pid in unique_pids[:20] if pid in train_pids])
print("Test  PIDs from first 20:", [pid for pid in unique_pids[:20] if pid in test_pids])

In [ ]:
print(f"Train PIDs: {len(train_pids)} | Rows: {len(train_df)} | saved -> {train_path}") 
print(f"Test PIDs: {len(test_pids)} | Rows: {len(test_df)} | saved -> {test_path}")

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Paths
src_path = Path(" # png-slice-information")          # change if needed
train_path = Path(" # train-png-slice-information")          # change if needed
test_path = Path(" # test-png-slice-information")          # change if needed

# Load
df = pd.read_csv(src_path)

pid_col = "pid"

# Unique PIDs with conversion check
pid_dtype = df[pid_col].dtype
print(f"Detected PID column: {pid_col} (dtype: {pid_dtype})")

# Only convert if dtype is numeric or if it looks like float-encoded IDs (e.g., 1234.0)
if np.issubdtype(pid_dtype, np.number):
    print("→ Numeric PID detected. Converting to string and cleaning possible '.0' suffixes.")
    pids_series = df[pid_col].dropna().astype(str).str.replace(r"\.0$", "", regex=True).str.strip()
else:
    # If already string-like, just clean whitespace and ensure no '.0' artifacts
    pids_series = df[pid_col].dropna().astype(str).str.strip().str.replace(r"\.0$", "", regex=True)

unique_pids = pids_series.unique()
n_unique = len(unique_pids)
print(f"Unique PIDs detected: {n_unique}")

# Desired counts
n_train, n_test = 670, 75
if n_unique < (n_train + n_test):
    raise ValueError(f"Not enough unique PIDs ({n_unique}) to split into {n_train} train and {n_test} test.")

# split: 85% train, 5% val, 10% test (PID-level)
block_size = 20
train_pids = set()
val_pids = set()
test_pids = set()

for i in range(0, len(unique_pids), block_size):
    block = unique_pids[i:i + block_size]

    # First 17 → train
    train_pids.update(block[:17])

    # 18th → val (if exists)
    if len(block) > 17:
        val_pids.add(block[17])

    # 19th & 20th → test (if exist)
    if len(block) > 18:
        test_pids.update(block[18:20])

pid_as_str = pids_series

train_df = df[pid_as_str.isin(train_pids)].copy()
val_df   = df[pid_as_str.isin(val_pids)].copy()
test_df  = df[pid_as_str.isin(test_pids)].copy()

train_df.to_csv(train_path, index=False)
val_df.to_csv("converted_slice_val.csv", index=False)
test_df.to_csv(test_path, index=False)

# Sanity checks

print(f"Train PIDs: {len(train_pids)} | Rows: {len(train_df)} | saved -> {train_path}")
print(f"Val   PIDs: {len(val_pids)}   | Rows: {len(val_df)}   | saved -> converted_slice_val.csv")
print(f"Test  PIDs: {len(test_pids)}  | Rows: {len(test_df)}  | saved -> {test_path}")